## Popcorn Hack 1: Check SFI Part Fields

Prediction: all original checks will be True. Changing the category
to Street Car will make only the accepted-category check False.

In [2]:
# CODE_RUNNER: Popcorn Hack 1 - Check SFI Part Fields

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]


def check_fields(record):
    # Whitespace-only text does not count as a completed field.
    print("Has product name:", bool(record.get("product_name", "").strip()))
    print("Accepted category:", record.get("category", "") in valid_categories)
    print("Has spec number:", bool(record.get("spec_number", "").strip()))
    print("Has effective date:", bool(record.get("effective_date", "").strip()))


print("Original record:")
check_fields(part)

changed_part = part.copy()
changed_part["category"] = "Street Car"

print("\nChanged category:")
check_fields(changed_part)

Original record:
Has product name: True
Accepted category: True
Has spec number: True
Has effective date: True

Changed category:
Has product name: True
Accepted category: False
Has spec number: True
Has effective date: True


## Popcorn Hack 2: Backend Create Rule

Prediction: the original record will return False because spec 1.1
already exists. Changing the spec to 1.2 will return True because
the required fields are valid and that spec is unused.

In [3]:
# CODE_RUNNER: Popcorn Hack 2 - Backend Create Rule

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]


def can_create(record):
    has_name = bool(record.get("product_name", "").strip())
    category_ok = record.get("category", "") in valid_categories
    has_spec = bool(record.get("spec_number", "").strip())
    has_date = bool(record.get("effective_date", "").strip())
    duplicate_spec = record.get("spec_number", "").strip() in existing_spec_numbers

    # Every requirement must pass, including not being a duplicate.
    return (
        has_name
        and category_ok
        and has_spec
        and has_date
        and not duplicate_spec
    )


print("Original record can be created:", can_create(part))

new_part = part.copy()
new_part["spec_number"] = "1.2"
print("New spec record can be created:", can_create(new_part))

Original record can be created: False
New spec record can be created: True


## Popcorn Hack 3: SFI Search Filter

Prediction: searching 1.2 will return one record. Searching flywheel
will return two records. Searching 9.9 will return no matches.
A matching record in the Street Car category will be excluded.

The query can match part of a product name or an exact spec number,
but the category must also be accepted.

In [4]:
# CODE_RUNNER: Popcorn Hack 3 - SFI Search Filter

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]


def search_records(query, data):
    query = query.strip().lower()
    matches = []

    for record in data:
        name_match = query in record["product_name"].lower()
        spec_match = query == record["spec_number"].lower()
        category_ok = record["category"] in valid_categories

        # Either search match is allowed; the category must also pass.
        if (name_match or spec_match) and category_ok:
            matches.append(record)

    return matches


def show_matches(query, data):
    print("Query:", query)
    matches = search_records(query, data)

    for record in matches:
        print(record["spec_number"], "-", record["product_name"])

    if not matches:
        print("No matches")


show_matches("1.2", records)
print()
show_matches("flywheel", records)
print()
show_matches("9.9", records)

invalid_record = {
    "product_name": "Street Flywheel",
    "category": "Street Car",
    "spec_number": "3.1"
}

print("\nCategory filtering test:")
show_matches("flywheel", records + [invalid_record])

Query: 1.2
1.2 - Multiple Disc Clutch Assemblies

Query: flywheel
1.1 - Replacement Flywheels and Clutch Assemblies
2.1 - Racing Flywheel Record

Query: 9.9
No matches

Category filtering test:
Query: flywheel
1.1 - Replacement Flywheels and Clutch Assemblies
2.1 - Racing Flywheel Record


## MCQ 3.05

1. **A — True:** `>=` includes equality.
2. **C — `and`:** both required conditions must pass.
3. **A — True:** `not` reverses False.
4. **B:** `or` accepts either allowed category.

MCQ 3.05: 4/4 | answers: A,C,A,B

Score: **100%**

## Homework Hack: SFI Backend Validator

A valid record needs a nonempty product name, an accepted category,
a nonempty spec number, a nonempty effective date, and an unused spec.

Predicted results:
- Record 1: Accepted.
- Record 2: Rejected because its category is not accepted.
- Record 3: Rejected because its spec already exists.
- Missing-field, whitespace-only, and missing-key tests: False.

In [5]:
# CODE_RUNNER: Homework Hack - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    }
]


def validate_record(record):
    """Return whether a candidate passes all SFI validation rules.

    Parameters:
        record: Dictionary containing string values for product_name,
                category, spec_number, and effective_date.

    Returns:
        Boolean: True when every rule passes; otherwise False.
        Missing keys and whitespace-only required text are rejected.
    """
    has_name = bool(record.get("product_name", "").strip())
    accepted_category = record.get("category", "") in valid_categories
    has_spec = bool(record.get("spec_number", "").strip())
    has_date = bool(record.get("effective_date", "").strip())
    duplicate_spec = record.get("spec_number", "").strip() in existing_spec_numbers

    # AND combines the requirements; NOT excludes duplicate specs.
    return (
        has_name
        and accepted_category
        and has_spec
        and has_date
        and not duplicate_spec
    )


for number, record in enumerate(test_records, start=1):
    decision = "Accepted" if validate_record(record) else "Rejected"
    print(f"Record {number}: {decision}")

# Change one field at a time to check each required-field rule.
for field in ["product_name", "spec_number", "effective_date"]:
    changed_record = test_records[0].copy()
    changed_record[field] = ""
    print(f"Missing {field} accepted:", validate_record(changed_record))

blank_name = test_records[0].copy()
blank_name["product_name"] = "   "
print("Whitespace-only name accepted:", validate_record(blank_name))

missing_key = test_records[0].copy()
del missing_key["effective_date"]
print("Missing date key accepted:", validate_record(missing_key))

Record 1: Accepted
Record 2: Rejected
Record 3: Rejected
Missing product_name accepted: False
Missing spec_number accepted: False
Missing effective_date accepted: False
Whitespace-only name accepted: False
Missing date key accepted: False


## Boolean Logic and Pseudocode Comparison

The validator uses `and` because every required field and rule must
pass. `not` prevents acceptance of a duplicate spec. The search uses
`or` because either a product-name match or an exact spec match is
sufficient, while the accepted-category requirement must also pass.

```text
isValid ← hasName AND acceptedCategory AND hasSpec AND hasDate AND NOT duplicateSpec
```

For the first homework record, every required field is present,
its category is accepted, and spec 1.2 is unused. The predicted
pseudocode result is True, matching Python's Record 1: Accepted.

The second record fails the category requirement. The third record
fails the uniqueness requirement.